In [3]:
# Kiểm tra nhanh các thư viện cơ bản (không bắt buộc tất cả phải có)
import sys, json, math, random, os, time
import numpy as np

try:
    import sklearn
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix
    SKLEARN_OK = True
except Exception as e:
    SKLEARN_OK = False
    print("Thiếu scikit-learn. Các phần dùng sklearn sẽ không chạy:", e)

try:
    import networkx as nx
    NETWORKX_OK = True
except Exception as e:
    NETWORKX_OK = False
    print("Thiếu networkx. Các phần đồ thị sẽ không chạy:", e)

# Các phần tùy chọn (có thể không có internet nên sẽ fail ở đây, cứ để False nếu không)
try:
    import torch
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False

try:
    import transformers
    TRANSFORMERS_OK = True
except Exception as e:
    TRANSFORMERS_OK = False

print("SKLEARN_OK =", SKLEARN_OK, "| NETWORKX_OK =", NETWORKX_OK, "| TORCH_OK =", TORCH_OK, "| TRANSFORMERS_OK =", TRANSFORMERS_OK)

SKLEARN_OK = True | NETWORKX_OK = True | TORCH_OK = True | TRANSFORMERS_OK = True


## 11) Self-Organizing Maps (SOM) – Ứng dụng: **Phân cụm khách hàng** (Iris ~ mô phỏng RFM)

**Ý tưởng:** SOM ánh xạ dữ liệu đa chiều về lưới 2D, giữ **cấu trúc topological**, hữu ích để **phân khúc khách hàng**.
Ta dùng bộ **Iris** (4 đặc trưng) như proxy dữ liệu RFM (đơn giản).

**Bước:** Chuẩn hóa → Train SOM 10x10 → Gán mỗi điểm vào Best Matching Unit (BMU) → Quan sát cụm.

In [4]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from collections import Counter

# SOM tối giản
class MiniSOM:
    def __init__(self, m, n, dim, lr=0.5, sigma=None, iters=2000, seed=0):
        self.m, self.n, self.dim = m, n, dim
        self.lr0 = lr
        self.sigma0 = sigma if sigma is not None else max(m,n)/2
        self.iters = iters
        rng = np.random.default_rng(seed)
        self.W = rng.normal(0,1,size=(m*n, dim))

    def _grid(self):
        # tọa độ lưới 2D cho mỗi neuron
        coords = np.array([(i,j) for i in range(self.m) for j in range(self.n)])
        return coords

    def _bmu(self, x):
        # tìm BMU (neuron gần nhất theo L2)
        d = ((self.W - x)**2).sum(axis=1)
        return np.argmin(d)

    def fit(self, X):
        coords = self._grid()
        for t in range(1, self.iters+1):
            x = X[np.random.randint(0, len(X))]
            bmu = self._bmu(x)
            # lịch lr & sigma giảm dần
            lr = self.lr0 * np.exp(-t/self.iters)
            sigma = self.sigma0 * np.exp(-t/self.iters)
            # ảnh hưởng theo khoảng cách lưới
            d2 = ((coords - coords[bmu])**2).sum(axis=1)
            h = np.exp(-d2/(2*sigma**2))  # [m*n]
            # cập nhật trọng số
            self.W += lr * h[:,None] * (x - self.W)

    def predict_bmu(self, X):
        return np.array([self._bmu(x) for x in X])

# Load dữ liệu
iris = load_iris()
X = iris['data']
y = iris['target']
sc = StandardScaler()
Xz = sc.fit_transform(X)

som = MiniSOM(m=10, n=10, dim=Xz.shape[1], lr=0.5, iters=3000, seed=42)
som.fit(Xz)
bmu_idx = som.predict_bmu(Xz)

# Thống kê mỗi cụm (BMU) chứa nhãn nào
counts = {}
for bmu, label in zip(bmu_idx, y):
    counts.setdefault(bmu, []).append(label)

cluster_summ = {k: dict(Counter(v)) for k,v in counts.items()}
# In top 10 BMU đông nhất
top = sorted(cluster_summ.items(), key=lambda kv: sum(kv[1].values()), reverse=True)[:10]
print("Top 10 BMU (neuron) đông nhất & phân bố nhãn:")
for nid, dist in top:
    print(f"BMU {nid:3d}: total={sum(dist.values()):3d} | {dist}")

Top 10 BMU (neuron) đông nhất & phân bố nhãn:
BMU   9: total= 10 | {np.int64(0): 10}
BMU  99: total=  8 | {np.int64(2): 8}
BMU  79: total=  8 | {np.int64(2): 8}
BMU   7: total=  7 | {np.int64(0): 7}
BMU   3: total=  7 | {np.int64(0): 7}
BMU   4: total=  7 | {np.int64(0): 7}
BMU   5: total=  5 | {np.int64(0): 5}
BMU  90: total=  5 | {np.int64(2): 5}
BMU  30: total=  4 | {np.int64(1): 3, np.int64(2): 1}
BMU  45: total=  4 | {np.int64(1): 4}
